# Lab 6 — Session & Context (Features #10–#15)
**AmbariCloud FDE Workshop · Day 1** · Work inside `~/workshop-todo`.

## Feature #10 — /compact: reclaim context, keep continuity
Summarises the conversation and continues with the summary. Optional focus: `/compact keep the failing test names and file paths`. Auto-compact fires when the window fills — but YOU pick what survives if you run it early (≈50–60%).
> ⌨️ **Interactive step — do this in a real terminal**, not in this notebook. Claude Code is a TUI; the notebook cells below only run *headless* (`claude -p`) commands.

```
> /context                                   # note % used
> /cost
> /compact focus on todo.py API and open bugs
> /context                                   # % drops
> what command were we adding?               # it still knows
```
**Watch for:** compacting mid-task loses exact error text and diffs — finish the step, THEN compact.

## Feature #11 — /clear: start fresh
Wipes conversation + input history; same folder, same CLAUDE.md. Previous conversation preserved on disk — resumable.
```
> /clear
> what were we working on?    # -> it does not know (good)
> /resume                     # pick earlier session -> history back
```
**Habit:** one task per context. `/clear` does NOT reset CLAUDE.md/rules/memory — only chat.

## Feature #12 — /context: how full is the window?
Shows system prompt, CLAUDE.md files, tools, messages, free space. 70% window → compact first.
```
> /context
> @todo.py @tests/test_todo.py @README.md
> /context        # messages grew
> /compact
> /context        # back down
```
**Watch for:** large tool outputs (cat of a log, full test run) consume context silently — pipe to head/tail.


## Feature #13 — Rewind (Esc Esc): Ctrl+Z for the session
Input EMPTY → press `Esc` twice → checkpoint menu. Restore: code only, conversation only, or both. Same menu via `/rewind`.
> ⌨️ **Interactive step — do this in a real terminal**, not in this notebook. Claude Code is a TUI; the notebook cells below only run *headless* (`claude -p`) commands.

```
> rewrite todo.py to use a class TodoStore     # let it finish; tests may break
# input empty -> Esc, Esc -> pick checkpoint before the rewrite
# choose 'Restore code and conversation'
> ! git diff --stat        # todo.py is back
```
**Watch for:** rewind restores files Claude edited — NOT shell side-effects (pip installs, git commits, `rm`). Git is still your safety net.

## Feature #14 — Checkpointing: automatic snapshots
Claude snapshots files BEFORE each edit — that's what Rewind restores. Per-session, stored in `~/.claude`, independent of git.
```
> add input validation to every command
> now add colour output with rich
# Esc Esc -> BOTH checkpoints listed -> restore to 'after validation' only
```
**Watch for:** checkpoints live with the session; `/clear` = new session = new checkpoint list. Commits are the durable history.

## Feature #15 — Resume / Continue: clean pickups
| Command | Does |
|---|---|
| `claude -c` | continue most recent session in THIS directory |
| `claude -r` | interactive session picker; `claude -r <id\|name>` jumps in |
| `/resume` `/rename` | same inside a session |
| `--fork-session` | resume into a NEW id (branch an old chat) |
```
> /rename todo-lab
> /exit
claude -c                # right back in
claude -r todo-lab       # by name
```
**Pro habit:** end long tasks with `> write HANDOFF.md: what is done, what is next, how to run tests` — a new session (or colleague) reads it in one `@`.
**Watch for:** sessions are per directory; `cd` elsewhere and `-c` finds nothing.


### Inspect your sessions from the shell (runnable)

In [ ]:
%%bash
ls ~/.claude/projects/ 2>/dev/null | head -5
echo "---"
claude -p "Say OK" --output-format json 2>/dev/null | python3 -c "import json,sys; d=json.load(sys.stdin); print('session_id:', d.get('session_id','?'), '| cost_usd:', d.get('total_cost_usd','?'))" 